# Mobile Addiction Classification - Preprocessing

This notebook contains the data preprocessing components including data cleaning, feature engineering, and train/test splitting.

### Bug Fix #2: `StringDtype` → `object`
Pandas 2.x reads CSV string columns as `StringDtype` which is incompatible with sklearn's `ColumnTransformer`.
We must convert them to `object` dtype explicitly.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

print("All imports successful")

All imports successful


## 1. Load and Clean Data

In [2]:
# Load data
data = pd.read_csv("../data/raw/mobile_addiction.csv")

# Drop unnamed index column if present
if data.columns[0].startswith('Unnamed'):
    data = data.drop(data.columns[0], axis=1)

print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

Shape: (13589, 11)
Columns: ['daily_screen_time', 'app_sessions', 'social_media_usage', 'gaming_time', 'notifications', 'night_usage', 'age', 'work_study_hours', 'stress_level', 'apps_installed', 'addicted']


,daily_screen_time,app_sessions,social_media_usage,gaming_time,notifications,night_usage,age,work_study_hours,stress_level,apps_installed,addicted
0,2,29,0,0,49,0,44,5,3,35,not addicted
1,6,29,1,2,65,1,29,5,9,21,addicted
2,9,28,2,0,57,3,28,7,5,39,addicted
3,6,39,2,0,69,1,28,6,8,24,addicted
4,5,37,3,1,64,2,27,4,5,26,addicted


In [3]:
# FIX: Convert StringDtype columns to object (Pandas 2.x compatibility)
# This resolves: TypeError: Cannot interpret '<StringDtype(na_value=nan)>' as a data type
for col in data.columns:
    if hasattr(data[col].dtype, 'name') and 'string' in str(data[col].dtype).lower():
        data[col] = data[col].astype('object')
    elif data[col].dtype == 'object':
        data[col] = data[col].astype('object')

print("Dtypes after fix:")
print(data.dtypes)

Dtypes after fix:
daily_screen_time      int64
app_sessions           int64
social_media_usage     int64
gaming_time            int64
notifications          int64
night_usage            int64
age                    int64
work_study_hours       int64
stress_level           int64
apps_installed         int64
addicted              object
dtype: object


## 2. Prepare Features and Target

In [4]:
# Define X and y
target = 'addicted'
X = data.drop(target, axis=1)

# Encode target: 'addicted' -> 1, 'not addicted' -> 0
y = data[target].map({'addicted': 1, 'not addicted': 0})

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")
print(f"Target encoding: {dict(y.value_counts())}")

X shape: (13589, 10)
y distribution:
addicted
1    6846
0    6743
Name: count, dtype: int64
Target encoding: {1: np.int64(6846), 0: np.int64(6743)}


In [5]:
# Train / test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution: {y_test.value_counts().to_dict()}")

Train size: 10871 | Test size: 2718
Train target distribution: {1: 5477, 0: 5394}
Test target distribution: {1: 1369, 0: 1349}


In [6]:
# Separate numeric and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric columns ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")

# Display basic statistics for numeric columns
if num_cols:
    print("\nNumeric columns statistics:")
    print(X_train[num_cols].describe().T)

Numeric columns (10): ['daily_screen_time', 'app_sessions', 'social_media_usage', 'gaming_time', 'notifications', 'night_usage', 'age', 'work_study_hours', 'stress_level', 'apps_installed']
Categorical columns (0): []

Numeric columns statistics:
                      count       mean        std   min   25%   50%   75%  \
daily_screen_time   10871.0   3.772054   1.895260   0.0   2.0   4.0   5.0   
app_sessions        10871.0  30.075338   7.398571   8.0  25.0  30.0  35.0   
social_media_usage  10871.0   1.545580   1.202618   0.0   1.0   1.0   2.0   
gaming_time         10871.0   1.037071   0.992152   0.0   0.0   1.0   2.0   
notifications       10871.0  60.019041  12.775951  25.0  50.0  59.0  70.0   
night_usage         10871.0   0.993561   0.950551   0.0   0.0   1.0   2.0   
age                 10871.0  33.032012  10.107685  15.0  25.0  33.0  41.0   
work_study_hours    10871.0   6.001472   2.075484   0.0   5.0   6.0   7.0   
stress_level        10871.0   4.260326   2.289275   0.0   3.

## 3. Build Preprocessing Pipeline

In [7]:
# Numeric pipeline
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

# Categorical pipeline
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ] if cat_cols else [
        ('num', num_transformer, num_cols)
    ]
)

print("Preprocessor built successfully")

Preprocessor built successfully


In [8]:
# Test the preprocessor on a small sample
print("Testing preprocessor on training data...")
X_train_processed = preprocessor.fit_transform(X_train.head(5))
print(f"Original shape: {X_train.head(5).shape}")
print(f"Processed shape: {X_train_processed.shape}")
print(f"Processed data type: {type(X_train_processed)}")

# If the result is a numpy array, show the first few values
if isinstance(X_train_processed, np.ndarray):
    print(f"Sample of processed data:\n{X_train_processed[:2, :5]}")
else:
    print(f"Processed columns: {X_train_processed.columns.tolist()[:5] if hasattr(X_train_processed, 'columns') else 'N/A'}")

Testing preprocessor on training data...
Original shape: (5, 10)
Processed shape: (5, 10)
Processed data type: <class 'numpy.ndarray'>
Sample of processed data:
[[0.         0.46666667 0.75       1.         1.        ]
 [0.33333333 0.86666667 0.75       0.         0.        ]]


## 4. Save Preprocessed Data for Training

In [9]:
# Process the full training and test sets
print("Processing full training and test sets...")

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Training data shape: {X_train_processed.shape}")
print(f"Test data shape: {X_test_processed.shape}")

# Convert to DataFrame if it's a numpy array
if isinstance(X_train_processed, np.ndarray):
    # Get feature names after preprocessing
    feature_names = []
    
    # Numeric features keep their names
    feature_names.extend(num_cols)
    
    # Categorical features get one-hot encoded names
    if cat_cols:
        for col in cat_cols:
            unique_values = X_train[col].unique()
            feature_names.extend([f"{col}_{val}" for val in unique_values])
    
    X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names[:X_train_processed.shape[1]])
    X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names[:X_test_processed.shape[1]])

print(f"Processed training data type: {type(X_train_processed)}")
print(f"Sample feature names: {X_train_processed.columns.tolist()[:10]}")

Processing full training and test sets...
Training data shape: (10871, 10)
Test data shape: (2718, 10)
Processed training data type: <class 'pandas.core.frame.DataFrame'>
Sample feature names: ['daily_screen_time', 'app_sessions', 'social_media_usage', 'gaming_time', 'notifications', 'night_usage', 'age', 'work_study_hours', 'stress_level', 'apps_installed']


In [10]:
# Save preprocessed data and preprocessor
import joblib

# Save preprocessed data
X_train_processed.to_csv('..\data\processed\X_train_processed.csv', index=False)
X_test_processed.to_csv('..\data\processed\X_test_processed.csv', index=False)
y_train.to_csv('..\data\processed\y_train.csv', index=False)
y_test.to_csv('..\data\processed\y_test.csv', index=False)

# Save preprocessor
joblib.dump(preprocessor, '..\data\processed\preprocessor.pkl')

# Save column information
column_info = {
    'numeric_columns': num_cols,
    'categorical_columns': cat_cols,
    'target_column': target
}
joblib.dump(column_info, '..\data\processed\column_info.pkl')

print("Preprocessing complete! Files saved:")
print("- X_train_processed.csv")
print("- X_test_processed.csv")
print("- y_train.csv")
print("- y_test.csv")
print("- preprocessor.pkl")
print("- column_info.pkl")

Preprocessing complete! Files saved:
- X_train_processed.csv
- X_test_processed.csv
- y_train.csv
- y_test.csv
- preprocessor.pkl
- column_info.pkl


In [11]:
# Verify saved data can be loaded correctly
print("Verifying saved data...")

# Load and check data
X_train_loaded = pd.read_csv('..\data\processed\X_train_processed.csv')
y_train_loaded = pd.read_csv('..\data\processed\y_train.csv').squeeze()
preprocessor_loaded = joblib.load('..\data\processed\preprocessor.pkl')
column_info_loaded = joblib.load('..\data\processed\column_info.pkl')

print(f"Loaded X_train shape: {X_train_loaded.shape}")
print(f"Loaded y_train shape: {y_train_loaded.shape}")
print(f"Column info: {column_info_loaded}")
print(f"Preprocessor type: {type(preprocessor_loaded)}")

print("\nPreprocessing verification successful!")

Verifying saved data...
Loaded X_train shape: (10871, 10)
Loaded y_train shape: (10871,)
Column info: {'numeric_columns': ['daily_screen_time', 'app_sessions', 'social_media_usage', 'gaming_time', 'notifications', 'night_usage', 'age', 'work_study_hours', 'stress_level', 'apps_installed'], 'categorical_columns': [], 'target_column': 'addicted'}
Preprocessor type: <class 'sklearn.compose._column_transformer.ColumnTransformer'>

Preprocessing verification successful!


## Preprocessing Summary

The preprocessing pipeline has been successfully created and applied:

### Key Steps Completed:
1. **Data Loading**: Loaded the mobile addiction dataset
2. **StringDtype Fix**: Converted Pandas 2.x StringDtype columns to object dtype for sklearn compatibility
3. **Target Encoding**: Mapped 'addicted' → 1, 'not addicted' → 0
4. **Train/Test Split**: Created stratified split (80/20) maintaining class balance
5. **Preprocessing Pipeline**: Built ColumnTransformer with:
   - Numeric features: median imputation + MinMax scaling
   - Categorical features: most frequent imputation + one-hot encoding
6. **Data Processing**: Applied preprocessing to train and test sets
7. **Data Persistence**: Saved all processed data and preprocessing objects

### Files Generated:
- `X_train_processed.csv` - Processed training features
- `X_test_processed.csv` - Processed test features  
- `y_train.csv` - Training target
- `y_test.csv` - Test target
- `preprocessor.pkl` - Fitted preprocessing pipeline
- `column_info.pkl` - Column metadata

The data is now ready for model training in the next notebook.